# IO Cloud Agent Cloud — Skills Tutorial

## From Raw Tools to Skills

| | Notebook 2: Raw Tools | Notebook 3: Skills |
|---|---|---|
| What the LLM sees | 18 raw MCP tools | 3 high-level Skills |
| User says | "Check hardware, then estimate price, then deploy" | "Deploy an nginx, cheapest option" |
| LLM call count | One LLM call per step | One call to pick a Skill, internal auto-chaining |
| Reliability | Depends on LLM choosing correctly each step | Deterministic logic inside Skills, more stable |

### This tutorial defines 3 Skills:

| Skill | Function | Chained MCP Tools |
|-------|----------|------|
| `hardware_scout` | Hardware reconnaissance | Query hardware catalog -> Sort by price -> Return recommendations |
| `smart_deploy` | Smart deployment | Query hardware -> Estimate price -> Deploy -> Confirm status |
| `deployment_manager` | Deployment manager | List deployments / Check status / Destroy |

## 0. Environment Setup

In [1]:
import os
# Uncomment the following two lines only if you need a local proxy to access the internet
# os.environ['http_proxy']  = 'http://127.0.0.1:7890'
# os.environ['https_proxy'] = 'http://127.0.0.1:7890'

In [2]:
import json, asyncio, time
from openai import OpenAI
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# IO Intelligence API key for calling GLM-5.1
# This is the Intelligence Key, not the Cloud/MCP key
INTELLIGENCE_KEY = 'io-v2-****'
LLM_BASE_URL = 'https://api.intelligence.io.solutions/api/v1'
LLM_MODEL = 'zai-org/GLM-5.1'

# IO Cloud MCP key for GPU infrastructure operations
# This is the Cloud Key, not the Intelligence key
CLOUD_KEY = 'io-v2-*****'
MCP_URL = 'https://mcp.io.solutions/mcp'
MCP_HEADERS = {'x-api-key': CLOUD_KEY}

llm = OpenAI(api_key=INTELLIGENCE_KEY, base_url=LLM_BASE_URL)
print('OK')

OK


## 1. MCP Connection Layer

In [3]:
async def call_mcp_tool(tool_name, arguments=None):
    """MCP tool invocation with retry."""
    for attempt in range(3):
        try:
            async with streamablehttp_client(MCP_URL, headers=MCP_HEADERS, timeout=60) as (r, w, _):
                async with ClientSession(r, w) as session:
                    await session.initialize()
                    result = await session.call_tool(tool_name, arguments=arguments or {})
                    text = result.content[0].text if result.content else ''
                    try:
                        return json.loads(text)
                    except json.JSONDecodeError:
                        return text
        except Exception as e:
            if attempt < 2:
                await asyncio.sleep(2)
            else:
                return {'error': str(e)}


def extract_data(resp):
    if isinstance(resp, dict) and 'data' in resp:
        inner = resp['data']
        if isinstance(inner, dict) and 'data' in inner:
            return inner['data']
        return inner
    return resp

print('MCP connection layer OK')

MCP connection layer OK


## 2. Skill Framework
Each Skill is a dictionary containing:
- `name`: Skill name
- `description`: Description (shown to the LLM)
- `parameters`: Input parameter schema
- `handler`: Actual execution function (internally chains multiple MCP tools)

In [4]:
# Skill registry
SKILLS = {}

def register_skill(name, description, parameters, handler):
    """Register a Skill."""
    SKILLS[name] = {
        'name': name,
        'description': description,
        'parameters': parameters,
        'handler': handler
    }

def skills_to_openai_tools():
    """Convert all Skills to OpenAI function calling format."""
    return [
        {
            'type': 'function',
            'function': {
                'name': s['name'],
                'description': s['description'],
                'parameters': s['parameters']
            }
        }
        for s in SKILLS.values()
    ]

print('Skill framework OK')

Skill framework OK


## 3. Skill: hardware_scout (Hardware Reconnaissance)

One sentence to query hardware; internally auto-chains: query catalog -> sort by price -> return Top N recommendations.

In [5]:
async def hardware_scout(gpu_filter=None, top_n=5):
    """
    Hardware reconnaissance Skill.
    Internal chaining: caas_get_hardware_ids -> sort & filter -> return recommendations
    """
    steps = []

    # Step 1: Query hardware catalog
    steps.append('[Step 1] Querying CaaS hardware catalog...')
    raw = await call_mcp_tool('caas_get_hardware_ids')
    items = extract_data(raw)

    # Handle nested structure
    if isinstance(items, dict):
        for v in items.values():
            if isinstance(v, list):
                items = v
                break

    if not isinstance(items, list):
        return {'steps': steps, 'error': 'Unable to parse hardware list', 'raw': str(items)[:500]}

    steps.append(f'  Found {len(items)} hardware types')

    # Step 2: Sort by price
    steps.append('[Step 2] Sorting by price...')
    sorted_items = sorted(items, key=lambda x: x.get('price', 999) if isinstance(x, dict) else 999)

    # Step 3: Optional GPU filter
    if gpu_filter:
        steps.append(f'[Step 3] Filtering GPUs containing "{gpu_filter}"...')
        sorted_items = [
            it for it in sorted_items
            if isinstance(it, dict) and gpu_filter.lower() in str(it.get('hardware_name', '')).lower()
        ]
        steps.append(f'  {len(sorted_items)} types remaining after filter')

    # Step 4: Return Top N
    top = sorted_items[:top_n]
    recommendations = []
    for it in top:
        if isinstance(it, dict):
            recommendations.append({
                'hardware_id': it.get('hardware_id'),
                'name': it.get('hardware_name'),
                'price_per_hr': it.get('price'),
                'available': it.get('available'),
                'location': it.get('location'),
                'max_gpus': it.get('max_gpus_per_container')
            })

    steps.append(f'[Done] Returning Top {len(recommendations)} recommendations')

    return {
        'steps': steps,
        'recommendations': recommendations,
        'total_hardware_count': len(items)
    }


register_skill(
    name='hardware_scout',
    description='Hardware reconnaissance: Query available GPU hardware, sort by price, return recommendation list. Optionally filter by GPU model.',
    parameters={
        'type': 'object',
        'properties': {
            'gpu_filter': {
                'type': 'string',
                'description': 'GPU model filter, e.g. "H100", "4090", "A100". Leave empty to return all.'
            },
            'top_n': {
                'type': 'integer',
                'description': 'Return the top N cheapest options, default 5',
                'default': 5
            }
        }
    },
    handler=hardware_scout
)

print('Skill hardware_scout registered')

Skill hardware_scout registered


## 4. Skill: smart_deploy (Smart Deployment)

One sentence to deploy; internally auto-chains: query hardware -> estimate price -> deploy -> confirm status.
> This Skill incurs real charges!

In [6]:
async def smart_deploy(image_url='nginx:latest', duration_hours=1, hardware_id=12, location_ids=None):
    """
    Smart deployment Skill.
    Internal chaining: estimate price -> deploy -> check status
    """
    if location_ids is None:
        location_ids = [2]  # Default: US

    steps = []

    # Step 1: Estimate price
    steps.append(f'[Step 1] Estimating price: hw_id={hardware_id}, {duration_hours}h...')
    price = await call_mcp_tool('caas_get_price_estimate', {
        'location_ids': location_ids,
        'hardware_id': hardware_id,
        'duration_hours': duration_hours,
        'gpus_per_container': 1,
        'replica_count': 1
    })
    price_data = extract_data(price)
    steps.append(f'  Price estimate: {json.dumps(price_data, ensure_ascii=False)[:200]}')

    # Step 2: Deploy
    deploy_name = f'skill-deploy-{int(time.time()) % 100000}'
    steps.append(f'[Step 2] Deploying container: {deploy_name}, image={image_url}...')
    deploy_result = await call_mcp_tool('caas_deploy_container', {
        'request': {
            'resource_private_name': deploy_name,
            'duration_hours': duration_hours,
            'gpus_per_container': 1,
            'hardware_id': hardware_id,
            'replica_count': 1,
            'traffic_port': 80,
            'image_url': image_url,
            'location_ids': location_ids
        }
    })

    dep_data = extract_data(deploy_result)
    deployment_id = None
    if isinstance(dep_data, dict):
        deployment_id = dep_data.get('id') or dep_data.get('deployment_id')

    if not deployment_id:
        steps.append(f'  Deployment may have failed: {json.dumps(deploy_result, ensure_ascii=False)[:300]}')
        return {'steps': steps, 'error': 'Failed to obtain deployment_id', 'raw': deploy_result}

    steps.append(f'  Deployment successful! ID: {deployment_id}')

    # Step 3: Check status
    steps.append('[Step 3] Checking deployment status...')
    status = await call_mcp_tool('caas_get_deployment', {
        'deployment_id': str(deployment_id)
    })
    status_data = extract_data(status)
    steps.append(f'  Status: {status_data.get("status", "unknown") if isinstance(status_data, dict) else "?"}')

    return {
        'steps': steps,
        'deployment_id': deployment_id,
        'deployment_name': deploy_name,
        'status': status_data
    }


register_skill(
    name='smart_deploy',
    description='Smart deployment: Automatically estimate price and deploy a container. Incurs real charges! Defaults to RTX 4090 (hw_id=12) in the US (loc=2).',
    parameters={
        'type': 'object',
        'properties': {
            'image_url': {
                'type': 'string',
                'description': 'Container image, e.g. nginx:latest',
                'default': 'nginx:latest'
            },
            'duration_hours': {
                'type': 'integer',
                'description': 'Deployment duration in hours, default 1',
                'default': 1
            },
            'hardware_id': {
                'type': 'integer',
                'description': 'Hardware ID, default 12 (RTX 4090)',
                'default': 12
            }
        }
    },
    handler=smart_deploy
)

print('Skill smart_deploy registered')

Skill smart_deploy registered


## 5. Skill: deployment_manager (Deployment Manager)
One sentence to manage deployments: list / check status / destroy.

In [7]:
async def deployment_manager(action='list', deployment_id=None):
    """
    Deployment manager Skill.
    action: list / status / destroy
    """
    steps = []

    if action == 'list':
        steps.append('[Step 1] Listing all CaaS deployments...')
        result = await call_mcp_tool('caas_list_deployments', {'page': 1, 'page_size': 10})
        data = extract_data(result)
        deployments = []
        if isinstance(data, dict) and 'deployments' in data:
            deployments = data['deployments']
        steps.append(f'  Found {len(deployments)} deployments')
        return {'steps': steps, 'deployments': deployments}

    elif action == 'status' and deployment_id:
        steps.append(f'[Step 1] Querying deployment {deployment_id} status...')
        result = await call_mcp_tool('caas_get_deployment', {'deployment_id': deployment_id})
        data = extract_data(result)
        steps.append(f'  Status: {data.get("status", "?") if isinstance(data, dict) else "?"}')

        steps.append('[Step 2] Querying container details...')
        containers = await call_mcp_tool('caas_get_deployment_containers', {'deployment_id': deployment_id})
        containers_data = extract_data(containers)

        return {'steps': steps, 'deployment': data, 'containers': containers_data}

    elif action == 'destroy' and deployment_id:
        steps.append(f'[Step 1] Destroying deployment {deployment_id}...')
        result = await call_mcp_tool('caas_destroy_deployment', {'deployment_id': deployment_id})
        steps.append('[Step 2] Confirming destruction status...')
        status = await call_mcp_tool('caas_get_deployment', {'deployment_id': deployment_id})
        status_data = extract_data(status)
        steps.append(f'  Status: {status_data.get("status", "?") if isinstance(status_data, dict) else "?"}')
        return {'steps': steps, 'result': result, 'final_status': status_data}

    else:
        return {'error': f'Unknown action: {action}. Supported: list/status/destroy'}


register_skill(
    name='deployment_manager',
    description='Deployment manager: Manage container deployments. action=list to list all deployments; action=status to check a specific deployment status; action=destroy to destroy a specific deployment.',
    parameters={
        'type': 'object',
        'properties': {
            'action': {
                'type': 'string',
                'enum': ['list', 'status', 'destroy'],
                'description': 'Action type'
            },
            'deployment_id': {
                'type': 'string',
                'description': 'Deployment ID, required for status and destroy actions'
            }
        },
        'required': ['action']
    },
    handler=deployment_manager
)

print('Skill deployment_manager registered')
print(f'\n{len(SKILLS)} Skills registered: {list(SKILLS.keys())}')

Skill deployment_manager registered

3 Skills registered: ['hardware_scout', 'smart_deploy', 'deployment_manager']


## 6. Skill-Level Agent Loop
The LLM only needs to choose which Skill; the Skill internally auto-chains multiple MCP tools.

In [8]:
async def skill_agent_chat(user_message, verbose=True):
    """
    Skill-level Agent loop:
    User input -> GLM-5.1 picks a Skill -> Execute Skill -> GLM-5.1 summarizes
    """
    openai_tools = skills_to_openai_tools()

    messages = [
        {'role': 'system', 'content': (
            'You are an IO Cloud GPU infrastructure management assistant. '
            'You have 3 high-level Skills available, each of which internally auto-chains multiple operations. '
            'Choose the appropriate Skill based on user intent and fill in the parameters. '
            'Reply in English.'
        )},
        {'role': 'user', 'content': user_message}
    ]

    if verbose:
        print(f'[USER] {user_message}')

    # Round 1: LLM picks a Skill
    resp = llm.chat.completions.create(
        model=LLM_MODEL,
        messages=messages,
        tools=openai_tools,
        max_tokens=2000
    )

    assistant_msg = resp.choices[0].message
    messages.append(assistant_msg)

    if assistant_msg.tool_calls:
        if verbose and assistant_msg.content:
            print(f'\n[GLM-5.1] {assistant_msg.content}')

        for tc in assistant_msg.tool_calls:
            skill_name = tc.function.name
            skill_args = json.loads(tc.function.arguments) if tc.function.arguments else {}

            if verbose:
                print(f'\n[SKILL] {skill_name}({json.dumps(skill_args, ensure_ascii=False)})')

            # Execute Skill
            if skill_name in SKILLS:
                handler = SKILLS[skill_name]['handler']
                result = await handler(**skill_args)
            else:
                result = {'error': f'Unknown Skill: {skill_name}'}

            if verbose:
                # Print execution steps
                if isinstance(result, dict) and 'steps' in result:
                    print()
                    for step in result['steps']:
                        print(f'  {step}')

            messages.append({
                'role': 'tool',
                'tool_call_id': tc.id,
                'content': json.dumps(result, ensure_ascii=False, default=str)
            })

        # Round 2: LLM summarizes
        resp2 = llm.chat.completions.create(
            model=LLM_MODEL,
            messages=messages,
            max_tokens=2000
        )
        final_answer = resp2.choices[0].message.content
    else:
        final_answer = assistant_msg.content

    if verbose:
        print(f'\n[ANSWER]\n{final_answer}')

    return final_answer

print('Skill Agent loop OK')

Skill Agent loop OK


## 7. Demo: Hardware Reconnaissance

One sentence triggers `hardware_scout`, which internally auto-queries the catalog + sorts + filters.

In [9]:
answer = await skill_agent_chat('Show me available GPUs. What are the 5 cheapest options?')

[USER] Show me available GPUs. What are the 5 cheapest options?

[SKILL] hardware_scout({"top_n": 5})

  [Step 1] Querying CaaS hardware catalog...
    Found 61 hardware types
  [Step 2] Sorting by price...
  [Done] Returning Top 5 recommendations

[ANSWER]
Here are the **5 cheapest available GPUs** out of 61 total hardware types in the catalog:

| # | GPU | Price/hr | Available | Max GPUs |
|---|-----|----------|-----------|----------|
| 1 | **RTX A4000** | $0.18 | 7 | 1 |
| 2 | **RTX A5000** | $0.22 | 1 | 1 |
| 3 | **GeForce RTX 3090** | $0.27 | 39 | 1 |
| 4 | **GeForce RTX 4090** | $0.30 | 31 | 1 |
| 5 | **A6000** | $0.43 | 1 | 1 |

**Quick highlights:**
- 🏆 **Best budget pick:** RTX A4000 at just **$0.18/hr** with 7 units available.
- 💰 **Best value for performance:** GeForce RTX 3090 at **$0.27/hr** with 39 units available — great availability and strong performance.
- ⚡ **Most powerful in this list:** GeForce RTX 4090 at **$0.30/hr** with 31 units available — excellent price-to-p

--Filter for a specific GPU:

In [10]:
answer = await skill_agent_chat('Are there any H100 GPUs available? How much do they cost?')

[USER] Are there any H100 GPUs available? How much do they cost?

[GLM-5.1] Let me check the availability and pricing of H100 GPUs for you!

[SKILL] hardware_scout({"gpu_filter": "H100", "top_n": 5})

  [Step 1] Querying CaaS hardware catalog...
    Found 61 hardware types
  [Step 2] Sorting by price...
  [Step 3] Filtering GPUs containing "H100"...
    5 types remaining after filter
  [Done] Returning Top 5 recommendations

[ANSWER]
Great news — there are H100 GPUs available! Here's a summary of what I found:

| # | GPU Type | Price/hr | Available | Location | Max GPUs |
|---|----------|----------|-----------|----------|----------|
| 1 | **H100 PCIe** | $1.00/hr | 2 units | — | 1 |
| 2 | **H100 80G PCIe** | $1.00/hr | 141 units | — | 1 |
| 3 | **H100** | $1.848/hr | 1 unit | US | 1 |
| 4 | **H100 80GB HBM3** | $2.09/hr | 120 units | — | 1 |
| 5 | **H100 (SXM5 x8)** | $18.48/hr | 1 cluster | US | 8 |

### Key Takeaways:
- **Best availability**: The **H100 80G PCIe** ($1.00/hr) has **14

---

## 8. Demo: View My Deployments

One sentence triggers `deployment_manager`.

In [ ]:
answer = await skill_agent_chat('List all container deployments under my account')

---

## 9. Demo: Smart Deployment (Costs Real Money!)

> Running this cell incurs real charges! Please destroy the deployment in the next step after completion.

In [ ]:
# Uncomment to run (costs real money)
# answer = await skill_agent_chat('Deploy an nginx for me, use the cheapest RTX 4090, 1 hour is fine')

---

## 10. Interactive Chat

Modify the question and try it out:

In [11]:
your_question = 'Are there any A100 GPUs available? How much per hour?'

answer = await skill_agent_chat(your_question)

[USER] Are there any A100 GPUs available? How much per hour?

[GLM-5.1] Let me check the available A100 GPU options for you!

[SKILL] hardware_scout({"gpu_filter": "A100"})

  [Step 1] Querying CaaS hardware catalog...
    Found 62 hardware types
  [Step 2] Sorting by price...
  [Step 3] Filtering GPUs containing "A100"...
    8 types remaining after filter
  [Done] Returning Top 5 recommendations

[ANSWER]
Great news — A100 GPUs are available! Here's a summary of the options:

| # | Hardware | GPU Count | Price/hr | Available | Location |
|---|----------|-----------|----------|-----------|----------|
| 1 | **A100 (SXM4)** | 1x | **$0.97/hr** | 1 instance | US |
| 2 | **DGX A100** | 1x | **$0.97/hr** | 1 instance | US |
| 3 | **A100 SXM4 80GB** | 1x | **$1.39/hr** | 9 instances | — |
| 4 | **A100 80GB PCIe** | 1x | **$1.39/hr** | 8 instances | — |
| 5 | **A100 (2x)** | 2x | **$1.89/hr** | 1 instance | US |

**Key Takeaways:**
- 🟢 **Best value:** The A100 SXM4 and DGX A100 at **$0.97/hr

---

## Summary: Raw Tools vs Skills

```
Notebook 2: Raw Tools                    Notebook 3: Skills
────────────────────────────────────────────────────────────────────────────────
User: "Check hardware"                User: "Deploy an nginx, cheapest option"
  |                                       |
  v                                       v
LLM -> caas_get_hardware_ids            LLM -> smart_deploy Skill
  |                                       |
  v                                       v
LLM -> "Which one?"                    Skill internally auto-chains:
  |                                       1. Estimate price
  v                                       2. Deploy
LLM -> caas_get_price_estimate            3. Check status
  |                                       |
  v                                       v
LLM -> caas_deploy_container            LLM -> "Deployed, cost $0.30"
  |
  v
LLM -> "Deployment complete"

4 LLM calls                            2 LLM calls
Each step can go wrong                 Deterministic logic inside Skill
```

### When to use Skills?

- You have a fixed multi-step workflow (query hardware -> estimate price -> deploy)
- You need higher reliability (fewer LLM decision points)
- You want a simpler user experience (one sentence does it all)

### When to use Raw Tools?

- Exploratory queries (unsure which API to call)
- Flexible combinations (workflow is not fixed)
- Development and debugging phase